# Global Home-Field Advantage: Travel, Crowds, and Market Pricing

**Question:** Is home advantage in football driven by travel burden, by crowds, or by something else, and do betting markets price it correctly?

**Data:** 153,274 matches across 27 leagues (11 European leagues from 2000, 16 global leagues from 2012), with closing odds from 2012 onward. Source: football-data.co.uk.

**Stack:** DuckDB (SQL), Python for downloading and reference-data validation.

**Headline findings**
1. Home advantage is universal: about 0.34 goals per match (2012+), identical across European and global leagues.
2. Crowds are the dominant driver: empty stadiums (COVID) cut home advantage by 0.136 goals (95% CI 0.100–0.172), about 40%.
3. Travel is a weak factor: it explains ~21% of between-league variation, shows no reliable within-league effect, and same-city derbies are the one consistent distance effect.
4. The market prices distance correctly, absorbed the COVID shock with about a one-quarter lag, and has been efficient since 2021.

**Run order:** setup → download → stage → validate reference data → schema → load → views → findings. *Runtime → Run all* rebuilds everything from raw files.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Work from a fresh clone of the project repo; raw downloads are cached on Drive
import os, shutil, glob
REPO_URL = 'https://github.com/TylerWichman/global-home-advantage.git'
PROJECT  = '/content/global-home-advantage'
RAW_CACHE = '/content/drive/MyDrive/global-home-advantage-data/raw'

if not os.path.isdir(f'{PROJECT}/.git'):
    !git clone -q {REPO_URL} {PROJECT}
else:
    !git -C {PROJECT} pull -q
os.chdir(PROJECT)

# One-time: seed the cache from the old Drive project folder if it has the CSVs
os.makedirs(RAW_CACHE, exist_ok=True)
OLD_RAW = '/content/drive/MyDrive/Tyler_Wichman_Portfolio/global-home-advantage-sql/data'
if not glob.glob(f'{RAW_CACHE}/*.csv') and glob.glob(f'{OLD_RAW}/*_*.csv'):
    for f in glob.glob(f'{OLD_RAW}/main_*.csv') + glob.glob(f'{OLD_RAW}/extra_*.csv'):
        shutil.copy(f, RAW_CACHE)

# data/raw points at the Drive cache, so downloads persist between sessions
os.makedirs('data', exist_ok=True)
if not os.path.islink('data/raw'):
    shutil.rmtree('data/raw', ignore_errors=True)
    os.symlink(RAW_CACHE, 'data/raw')
for d in ['sql', 'python', 'docs', 'data/reference']:
    os.makedirs(d, exist_ok=True)
print(os.getcwd(), '|', len(glob.glob('data/raw/*.csv')), 'cached raw files')

In [ ]:
!pip install -q duckdb pandas numpy requests

In [ ]:
with open('.gitignore', 'w') as f:
    f.write("data/raw/\n*.duckdb\n*.duckdb.wal\n.ipynb_checkpoints/\n__pycache__/\ncities500.*\n")

import duckdb, glob
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Rebuilt from scratch each run (takes under a minute), so it lives in the clone, not on Drive
con = duckdb.connect('project.duckdb')

## 2. Download raw data
Main leagues are one file per league-season; extra leagues are one all-history file per league. Existing files are skipped.

In [ ]:
%%writefile python/download_data.py
"""Download Main League (season-by-season) and Extra League (all-history) files."""
import time
from pathlib import Path

import requests

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MAIN_LEAGUES = ["E0", "SC0", "D1", "I1", "SP1", "F1", "N1", "B1", "P1", "T1", "G1"]
FIRST_SEASON_START = 2000
LAST_SEASON_START = 2026

EXTRA_LEAGUES = [
    "ARG", "AUT", "BRA", "CHN", "DNK", "FIN", "IRL", "JPN",
    "MEX", "NOR", "POL", "ROU", "RUS", "SWE", "SWZ", "USA",
]


def season_codes(first, last):
    return [f"{str(y)[-2:]}{str(y + 1)[-2:]}" for y in range(first, last + 1)]


def save_as_utf8(content, path):
    try:
        text = content.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = content.decode("latin-1")
    path.write_text(text, encoding="utf-8")


def download(url, path):
    if path.exists():
        print(f"skip (exists): {path.name}")
        return
    resp = requests.get(url, timeout=60)
    if resp.status_code != 200:
        print(f"FAILED {resp.status_code}: {url}")
        return
    save_as_utf8(resp.content, path)
    print(f"saved: {path.name}")
    time.sleep(1)


def main():
    for league in MAIN_LEAGUES:
        for code in season_codes(FIRST_SEASON_START, LAST_SEASON_START):
            url = f"https://www.football-data.co.uk/mmz4281/{code}/{league}.csv"
            download(url, DATA_DIR / f"main_{league}_{code}.csv")

    for code in EXTRA_LEAGUES:
        url = f"https://football-data.co.uk/new/{code}.csv"
        download(url, DATA_DIR / f"extra_{code}.csv")


if __name__ == "__main__":
    main()

In [ ]:
!python python/download_data.py | grep -v "skip (exists)" | tail -5
n_main, n_extra = len(glob.glob('data/raw/main_*.csv')), len(glob.glob('data/raw/extra_*.csv'))
print(n_main, "main files |", n_extra, "extra files")
assert (n_main, n_extra) == (297, 16)

## 3. Stage raw CSVs
Everything is read as text; typing happens in the load step.

`null_padding = true` matters: many 2001–2005 files omit trailing empty odds columns on some rows. Without padding, `ignore_errors` silently discarded ~2,900 matches (e.g. France 2004/05 kept 15 of 380).

In [ ]:
%%writefile sql/01_staging.sql
-- null_padding fills rows that omit trailing empty columns (common in 2001-2005 files).
CREATE OR REPLACE TABLE stg_main AS
SELECT
    regexp_extract(filename, 'main_([A-Z0-9]+)_(\d{4})\.csv', 1) AS league_code,
    regexp_extract(filename, 'main_([A-Z0-9]+)_(\d{4})\.csv', 2) AS season_code,
    *
FROM read_csv('data/raw/main_*.csv', union_by_name = true, filename = true,
              all_varchar = true, null_padding = true, ignore_errors = true);

CREATE OR REPLACE TABLE stg_extra AS
SELECT
    regexp_extract(filename, 'extra_([A-Z0-9]+)\.csv', 1) AS league_code,
    *
FROM read_csv('data/raw/extra_*.csv', union_by_name = true, filename = true,
              all_varchar = true, null_padding = true, ignore_errors = true);

In [ ]:
con.execute(open('sql/01_staging.sql').read())
n = con.sql("SELECT (SELECT COUNT(*) FROM stg_main), (SELECT COUNT(*) FROM stg_extra)").fetchone()
print(n)
assert n == (90252, 63185)

In [ ]:
# Every raw data line must be staged (guards against silent row drops)
raw = []
for f in glob.glob('data/raw/main_*.csv') + glob.glob('data/raw/extra_*.csv'):
    with open(f, encoding='utf-8', newline='') as fh:
        raw.append((f, sum(1 for _ in fh) - 1))
con.register('raw_df', pd.DataFrame(raw, columns=['filename', 'raw_lines']))
off = con.sql("""
WITH staged AS (SELECT filename, COUNT(*) n FROM stg_main GROUP BY 1
                UNION ALL SELECT filename, COUNT(*) FROM stg_extra GROUP BY 1)
SELECT r.filename, r.raw_lines, COALESCE(s.n, 0) AS staged
FROM raw_df r LEFT JOIN staged s USING (filename)
WHERE r.raw_lines <> COALESCE(s.n, 0)
""").df()
print(len(off), "files with dropped rows")
assert off.empty, off

In [ ]:
%%writefile docs/profiling_notes.md
# Profiling Notes

## Main leagues (stg_main)
- 90,252 rows from 297 files (11 leagues x 27 seasons).
- Early-2000s files have rows missing trailing empty columns. Staging uses
  `null_padding`; without it ~2,900 rows were silently dropped.
- Older files store team names in HT/AT instead of HomeTeam/AwayTeam
  (948 rows). Normalization coalesces the two.
- 111 fully blank rows (no date, no teams) are dropped in normalization.
- Season comes from the file name (`0405` -> 2004/2005).
- Dates mix dd/mm/yy and dd/mm/yyyy; parsed by string length.
- Team names have trailing whitespace in places ('Ajax ' vs 'Ajax'); all
  joins use TRIM().
- Auto-named columns (column24..column121) are empty artifacts.

## Extra leagues (stg_extra)
- 63,185 rows from 16 files. China starts in 2014, the rest in 2012.
- Season is either "YYYY" (calendar-year leagues) or "YYYY/YYYY";
  start_year = first four characters.

## Odds
- Only closing odds are loaded (PSC*, AvgC*, B365C*), so odds-based
  analysis covers 2012 onward.
- Pinnacle closing odds stop partway through 2025; the analysis falls
  back to average closing odds after that.
- 26 average-odds rows with impossible overrounds (<1.0 or >1.25) are
  removed, mostly April-May 2026.

## Known data notes
- One ARG 2013/14 match was played in January 2015 (rescheduled); kept.
- G1 2001/02 had 14 teams (182 matches), which is a complete season.


## 4. Reference data (built once, validated on every run)
Three committed CSVs map raw team names to geocoded home cities. They involved manual research, so they are **validated here, not rebuilt**.

| File | Rows | Contents |
|---|---|---|
| `data/reference/leagues_seed.csv` | 27 | league code, name, country, source |
| `data/reference/ref_team_cities.csv` | 966 | raw team spelling → city + ISO2 country |
| `data/reference/ref_cities_coords.csv` | 742 | city → latitude/longitude + source |

**How they were built**
- **Current rosters (463 names):** each league's current-season Wikipedia club/location table, fuzzy-matched (rapidfuzz, score ≥ 80) to staged team names, with manual overrides for leagues the parser couldn't handle (`python/build_team_cities.py`, `python/consolidate_cities.py`, `data/reference/ref_team_cities_manual.csv`). Geocoding club names directly (Nominatim, Wikidata) was tried and abandoned because club names aren't indexed as places.
- **Historical and relegated clubs (503 names):** researched by hand (`data/reference/ref_team_cities_supplement.csv`), so every match in the dataset has a location.
- **Geocoding:** city names matched to GeoNames `cities500` on (name, country), preferring primary names and larger populations; 19 ambiguous or missing cities set manually.
- **Cleaning rules:** districts collapsed to their city (Beşiktaş → Istanbul, Anderlecht → Brussels); `country` is the city's own country, so Toronto is CA and Monaco is MC even though they play in other leagues; clubs that played top-flight home games elsewhere are mapped to where they played (Gretna → Motherwell, Evian → Annecy, Arles → Avignon, Tosno → Saint Petersburg).

In [ ]:
leagues = pd.read_csv('data/reference/leagues_seed.csv')
teams   = pd.read_csv('data/reference/ref_team_cities.csv')
cities  = pd.read_csv('data/reference/ref_cities_coords.csv')

assert len(leagues) == 27 and leagues['data_source'].value_counts().to_dict() == {'extra': 16, 'main': 11}
assert len(teams) == 966 and not teams.duplicated(['league_code', 'team']).any()
assert len(cities) == 742 and not cities.duplicated(['city', 'iso2']).any()
assert cities[['latitude', 'longitude']].notna().all().all()
assert set(teams['league_code']) == set(leagues['league_id'])

# Every team's (city, country) exists in the coordinates file
city_ids = set(cities['city'].str.replace(' ', '_') + '_' + cities['iso2'])
assert (teams['city'].str.replace(' ', '_') + '_' + teams['iso2']).isin(city_ids).all()

# Every coordinate falls inside its country's bounding box
BOX = {  # lat_min, lat_max, lon_min, lon_max (ES/PT/JP widened for islands, FI for Aland)
 'GB':(49.8,60.9,-8.7,1.8),'DE':(47.2,55.1,5.8,15.1),'IT':(36.6,47.1,6.6,18.6),
 'ES':(27.6,43.8,-18.2,4.4),'FR':(41.3,51.1,-5.2,9.6),'NL':(50.7,53.6,3.3,7.3),
 'BE':(49.5,51.6,2.5,6.4),'PT':(32.6,42.2,-31.3,-6.2),'TR':(35.8,42.1,25.6,44.8),
 'GR':(34.8,41.8,19.3,29.7),'AR':(-55.1,-21.8,-73.6,-53.6),'AT':(46.4,49.0,9.5,17.2),
 'BR':(-33.8,5.3,-74.0,-34.8),'CN':(18.2,53.6,73.5,134.8),'DK':(54.5,57.8,8.0,15.2),
 'FI':(59.8,70.1,19.0,31.6),'IE':(51.4,55.4,-10.5,-6.0),'JP':(24.0,45.6,122.9,146.0),
 'MX':(14.5,32.7,-118.4,-86.7),'NO':(57.9,71.2,4.6,31.1),'PL':(49.0,54.9,14.1,24.2),
 'RO':(43.6,48.3,20.2,29.7),'RU':(41.2,81.9,19.6,180.0),'SE':(55.3,69.1,11.0,24.2),
 'CH':(45.8,47.8,5.9,10.5),'US':(24.5,49.5,-125.0,-66.9),'CA':(41.7,83.1,-141.0,-52.6),
 'MC':(43.72,43.76,7.40,7.44),'LI':(47.04,47.28,9.47,9.64)}
in_box = cities.apply(lambda r: r.iso2 in BOX
                      and BOX[r.iso2][0] <= r.latitude <= BOX[r.iso2][1]
                      and BOX[r.iso2][2] <= r.longitude <= BOX[r.iso2][3], axis=1)
assert in_box.all(), cities[~in_box]

# Known distances within 3%
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    h = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(h))

pt = cities.set_index(['city', 'iso2'])
for (a, ia), (b, ib), km in [(('London','GB'), ('Manchester','GB'), 262),
                             (('Madrid','ES'), ('Barcelona','ES'), 505),
                             (('Milan','IT'), ('Rome','IT'), 477),
                             (('Istanbul','TR'), ('Ankara','TR'), 351),
                             (('Toronto','CA'), ('Vancouver','CA'), 3360)]:
    d = haversine_km(*pt.loc[(a, ia), ['latitude','longitude']], *pt.loc[(b, ib), ['latitude','longitude']])
    assert abs(d - km) / km < 0.03, (a, b, round(d))
print("Reference data valid:", cities['source'].value_counts().to_dict())

## 5. Schema
Six tables. `team_id` is league-scoped (`E0:Arsenal`), so a club that changes leagues gets one ID per league, which suits a per-league analysis. Fair probabilities are derived in the view, not stored.

In [ ]:
%%writefile sql/03_schema.sql
DROP TABLE IF EXISTS odds;
DROP TABLE IF EXISTS matches;
DROP TABLE IF EXISTS team_aliases;
DROP TABLE IF EXISTS teams;
DROP TABLE IF EXISTS cities;
DROP TABLE IF EXISTS leagues;

CREATE TABLE leagues (
    league_id   VARCHAR PRIMARY KEY,
    league_name VARCHAR NOT NULL,
    country     VARCHAR NOT NULL,
    data_source VARCHAR NOT NULL CHECK (data_source IN ('main', 'extra'))
);

CREATE TABLE cities (
    city_id   VARCHAR PRIMARY KEY,      -- e.g. 'Toronto_CA'
    city_name VARCHAR NOT NULL,
    country   VARCHAR NOT NULL,         -- ISO2 of the city itself, not the league
    latitude  DOUBLE NOT NULL CHECK (latitude BETWEEN -90 AND 90),
    longitude DOUBLE NOT NULL CHECK (longitude BETWEEN -180 AND 180),
    UNIQUE (city_name, country)
);

CREATE TABLE teams (
    team_id        VARCHAR PRIMARY KEY,   -- e.g. 'E0:Arsenal' (league-scoped by design)
    league_id      VARCHAR NOT NULL REFERENCES leagues (league_id),
    canonical_name VARCHAR NOT NULL,
    city_id        VARCHAR NOT NULL REFERENCES cities (city_id)
);

CREATE TABLE team_aliases (
    league_id VARCHAR NOT NULL REFERENCES leagues (league_id),
    alias     VARCHAR NOT NULL,
    team_id   VARCHAR NOT NULL REFERENCES teams (team_id),
    PRIMARY KEY (league_id, alias)
);

CREATE TABLE matches (
    match_id     VARCHAR PRIMARY KEY,
    league_id    VARCHAR NOT NULL REFERENCES leagues (league_id),
    season_label VARCHAR NOT NULL,
    start_year   INTEGER NOT NULL CHECK (start_year BETWEEN 1990 AND 2030),
    match_date   DATE NOT NULL,
    home_team_id VARCHAR NOT NULL REFERENCES teams (team_id),
    away_team_id VARCHAR NOT NULL REFERENCES teams (team_id),
    home_goals   INTEGER NOT NULL CHECK (home_goals >= 0),
    away_goals   INTEGER NOT NULL CHECK (away_goals >= 0),
    CHECK (home_team_id <> away_team_id),
    UNIQUE (league_id, match_date, home_team_id, away_team_id)
);

-- Fair probabilities are computed in the Phase 8 view, not stored.
CREATE TABLE odds (
    match_id  VARCHAR NOT NULL REFERENCES matches (match_id),
    bookmaker VARCHAR NOT NULL,
    odds_home DOUBLE CHECK (odds_home > 1),
    odds_draw DOUBLE CHECK (odds_draw > 1),
    odds_away DOUBLE CHECK (odds_away > 1),
    PRIMARY KEY (match_id, bookmaker)
);

## 6. Load
Normalizes both sources into `stg_all`, then loads teams, aliases, matches, and odds. Only spelling variants are merged into one team; renamed clubs (e.g. Guangzhou Evergrande → Guangzhou FC) keep separate IDs.

In [ ]:
%%writefile sql/04_load.sql
-- Rebuilds reference + fact tables from staged data and reference CSVs. Rerunnable.
DELETE FROM odds; DELETE FROM matches; DELETE FROM team_aliases;
DELETE FROM teams; DELETE FROM cities; DELETE FROM leagues;

-- 1. Reference tables
INSERT INTO leagues
SELECT TRIM(league_id), TRIM(league_name), TRIM(country), TRIM(data_source)
FROM read_csv('data/reference/leagues_seed.csv', header = true, all_varchar = true);

INSERT INTO cities
SELECT replace(TRIM(city), ' ', '_') || '_' || TRIM(iso2), TRIM(city), TRIM(iso2),
       CAST(latitude AS DOUBLE), CAST(longitude AS DOUBLE)
FROM read_csv('data/reference/ref_cities_coords.csv', header = true, all_varchar = true);

-- 2. Normalize both sources
CREATE OR REPLACE TABLE stg_all_raw AS
SELECT 'main' AS src,
       TRIM(league_code) AS league_id,
       '20' || LEFT(season_code, 2) || '/20' || RIGHT(season_code, 2) AS season_label,
       2000 + TRY_CAST(LEFT(season_code, 2) AS INTEGER) AS start_year,
       CASE LENGTH(TRIM("Date"))
            WHEN 8 THEN TRY_STRPTIME(TRIM("Date"), '%d/%m/%y')
            ELSE        TRY_STRPTIME(TRIM("Date"), '%d/%m/%Y') END::DATE AS match_date,
       COALESCE(NULLIF(TRIM(HomeTeam), ''), NULLIF(TRIM("HT"), '')) AS home_raw,
       COALESCE(NULLIF(TRIM(AwayTeam), ''), NULLIF(TRIM("AT"), '')) AS away_raw,
       TRY_CAST(TRY_CAST(TRIM(FTHG) AS DOUBLE) AS INTEGER) AS home_goals,
       TRY_CAST(TRY_CAST(TRIM(FTAG) AS DOUBLE) AS INTEGER) AS away_goals,
       TRY_CAST(TRIM(PSCH)   AS DOUBLE) AS pinnacle_close_h,
       TRY_CAST(TRIM(PSCD)   AS DOUBLE) AS pinnacle_close_d,
       TRY_CAST(TRIM(PSCA)   AS DOUBLE) AS pinnacle_close_a,
       TRY_CAST(TRIM(AvgCH)  AS DOUBLE) AS avg_close_h,
       TRY_CAST(TRIM(AvgCD)  AS DOUBLE) AS avg_close_d,
       TRY_CAST(TRIM(AvgCA)  AS DOUBLE) AS avg_close_a,
       TRY_CAST(TRIM(B365CH) AS DOUBLE) AS b365_close_h,
       TRY_CAST(TRIM(B365CD) AS DOUBLE) AS b365_close_d,
       TRY_CAST(TRIM(B365CA) AS DOUBLE) AS b365_close_a
FROM stg_main
UNION ALL
SELECT 'extra', TRIM(league_code), TRIM(Season),
       TRY_CAST(LEFT(TRIM(Season), 4) AS INTEGER),
       TRY_STRPTIME(TRIM("Date"), '%d/%m/%Y')::DATE,
       NULLIF(TRIM(Home), ''), NULLIF(TRIM(Away), ''),
       TRY_CAST(TRY_CAST(TRIM(HG) AS DOUBLE) AS INTEGER),
       TRY_CAST(TRY_CAST(TRIM(AG) AS DOUBLE) AS INTEGER),
       TRY_CAST(TRIM(PSCH) AS DOUBLE), TRY_CAST(TRIM(PSCD) AS DOUBLE), TRY_CAST(TRIM(PSCA) AS DOUBLE),
       TRY_CAST(TRIM(AvgCH) AS DOUBLE), TRY_CAST(TRIM(AvgCD) AS DOUBLE), TRY_CAST(TRIM(AvgCA) AS DOUBLE),
       TRY_CAST(TRIM(B365CH) AS DOUBLE), TRY_CAST(TRIM(B365CD) AS DOUBLE), TRY_CAST(TRIM(B365CA) AS DOUBLE)
FROM stg_extra;

CREATE OR REPLACE TABLE stg_all AS
SELECT md5(concat_ws('|', league_id, match_date, home_raw, away_raw)) AS match_id, *
FROM stg_all_raw
WHERE match_date IS NOT NULL AND start_year IS NOT NULL
  AND home_raw IS NOT NULL AND away_raw IS NOT NULL
  AND home_goals IS NOT NULL AND away_goals IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY league_id, match_date, home_raw, away_raw
                           ORDER BY src, season_label) = 1;

-- 3. Teams and aliases (spelling variants only; renamed clubs stay separate)
CREATE OR REPLACE TEMP TABLE ref_map AS
WITH ref AS (
    SELECT TRIM(league_code) AS league_code, TRIM(team) AS team,
           TRIM(city) AS city, TRIM(iso2) AS iso2
    FROM read_csv('data/reference/ref_team_cities.csv', header = true, all_varchar = true)
),
alias_merge(league_code, alias, canonical) AS (VALUES
    ('ARG', 'Colon Santa FE',   'Colon Santa Fe'),
    ('N1',  'Roda',             'Roda JC'),
    ('SWE', 'Oster',            'Osters'),
    ('G1',  'Yiannina',         'Giannina'),
    ('G1',  'Kalithea',         'Kallithea'),
    ('G1',  'Athens Kallithea', 'Kallithea'),
    ('POL', 'Ruch',             'Ruch Chorzow'),
    ('NOR', 'Ham-Kam',          'HamKam')
)
SELECT r.league_code, r.team,
       COALESCE(a.canonical, r.team) AS canonical,
       r.league_code || ':' || COALESCE(a.canonical, r.team) AS team_id,
       replace(r.city, ' ', '_') || '_' || r.iso2 AS city_id
FROM ref r
LEFT JOIN alias_merge a ON a.league_code = r.league_code AND a.alias = r.team;

INSERT INTO teams
SELECT DISTINCT team_id, league_code, canonical, city_id FROM ref_map;

INSERT INTO team_aliases
SELECT league_code, team, team_id FROM ref_map;

-- 4. Matches
INSERT INTO matches
SELECT s.match_id, s.league_id, s.season_label, s.start_year, s.match_date,
       ah.team_id, aa.team_id, s.home_goals, s.away_goals
FROM stg_all s
JOIN team_aliases ah ON ah.league_id = s.league_id AND ah.alias = s.home_raw
JOIN team_aliases aa ON aa.league_id = s.league_id AND aa.alias = s.away_raw;

-- 5. Closing odds, then drop impossible overrounds
INSERT INTO odds SELECT match_id, 'pinnacle_close', pinnacle_close_h, pinnacle_close_d, pinnacle_close_a
FROM stg_all WHERE pinnacle_close_h > 1 AND pinnacle_close_d > 1 AND pinnacle_close_a > 1;
INSERT INTO odds SELECT match_id, 'avg_close', avg_close_h, avg_close_d, avg_close_a
FROM stg_all WHERE avg_close_h > 1 AND avg_close_d > 1 AND avg_close_a > 1;
INSERT INTO odds SELECT match_id, 'b365_close', b365_close_h, b365_close_d, b365_close_a
FROM stg_all WHERE b365_close_h > 1 AND b365_close_d > 1 AND b365_close_a > 1;

DELETE FROM odds
WHERE 1/odds_home + 1/odds_draw + 1/odds_away NOT BETWEEN 1.0 AND 1.25;

In [ ]:
for f in ['sql/03_schema.sql', 'sql/04_load.sql']:
    con.execute(open(f).read())

counts = con.sql("""SELECT (SELECT COUNT(*) FROM leagues), (SELECT COUNT(*) FROM cities),
    (SELECT COUNT(*) FROM teams), (SELECT COUNT(*) FROM team_aliases),
    (SELECT COUNT(*) FROM stg_all), (SELECT COUNT(*) FROM matches), (SELECT COUNT(*) FROM odds)""").fetchone()
print(dict(zip(['leagues','cities','teams','aliases','stg_all','matches','odds'], counts)))
assert counts[4] == counts[5], "Matches dropped in load: unmapped team names"
assert counts == (27, 742, 958, 966, 153274, 153274, 223476)

In [ ]:
# Sanity: home-win share per league (expect roughly 0.40-0.50) and odds coverage by season
display(con.sql("""
SELECT m.league_id, l.country, COUNT(*) n,
       ROUND(AVG((home_goals > away_goals)::INT), 3) home_win,
       ROUND(AVG((home_goals < away_goals)::INT), 3) away_win,
       ROUND(AVG(home_goals - away_goals), 3) avg_gd
FROM matches m JOIN leagues l USING (league_id)
GROUP BY 1, 2 ORDER BY home_win DESC
""").df())
display(con.sql("""
SELECT m.start_year, COUNT(*) matches,
       ROUND(AVG((p.match_id IS NOT NULL)::INT), 3) pinnacle_cov,
       ROUND(AVG((a.match_id IS NOT NULL)::INT), 3) avg_cov
FROM matches m
LEFT JOIN odds p ON p.match_id = m.match_id AND p.bookmaker = 'pinnacle_close'
LEFT JOIN odds a ON a.match_id = m.match_id AND a.bookmaker = 'avg_close'
GROUP BY 1 ORDER BY 1
""").df())

## 7. Views
`match_facts` is the analysis base: one row per match with haversine travel distance, same-city and cross-border flags, result, and margin-free probabilities (Pinnacle closing odds, falling back to the market average).

In [ ]:
%%writefile sql/05_views.sql
CREATE OR REPLACE VIEW match_facts AS
WITH o AS (
    SELECT m.match_id,
           COALESCE(p.odds_home, a.odds_home) AS oh,
           COALESCE(p.odds_draw, a.odds_draw) AS od,
           COALESCE(p.odds_away, a.odds_away) AS oa,
           CASE WHEN p.match_id IS NOT NULL THEN 'pinnacle_close'
                WHEN a.match_id IS NOT NULL THEN 'avg_close' END AS odds_source
    FROM matches m
    LEFT JOIN odds p ON p.match_id = m.match_id AND p.bookmaker = 'pinnacle_close'
    LEFT JOIN odds a ON a.match_id = m.match_id AND a.bookmaker = 'avg_close'
)
SELECT
    m.match_id, m.league_id, l.country AS league_country, l.data_source,
    m.season_label, m.start_year, m.match_date,
    m.home_team_id, th.canonical_name AS home_team, ch.city_name AS home_city, ch.country AS home_iso2,
    m.away_team_id, ta.canonical_name AS away_team, ca.city_name AS away_city, ca.country AS away_iso2,
    m.home_goals, m.away_goals,
    m.home_goals - m.away_goals AS goal_diff,
    CASE WHEN m.home_goals > m.away_goals THEN 'H'
         WHEN m.home_goals = m.away_goals THEN 'D' ELSE 'A' END AS result,
    -- Haversine distance, Earth radius 6371 km
    2 * 6371 * ASIN(SQRT(
        POWER(SIN(RADIANS(ca.latitude - ch.latitude) / 2), 2)
      + COS(RADIANS(ch.latitude)) * COS(RADIANS(ca.latitude))
      * POWER(SIN(RADIANS(ca.longitude - ch.longitude) / 2), 2)
    )) AS travel_km,
    th.city_id = ta.city_id AS same_city,
    ch.country <> ca.country AS cross_border,
    o.odds_source,
    -- Fair probabilities (margin removed by normalizing the implied probabilities)
    (1/o.oh) / (1/o.oh + 1/o.od + 1/o.oa) AS prob_home_fair,
    (1/o.od) / (1/o.oh + 1/o.od + 1/o.oa) AS prob_draw_fair,
    (1/o.oa) / (1/o.oh + 1/o.od + 1/o.oa) AS prob_away_fair
FROM matches m
JOIN leagues l  ON l.league_id = m.league_id
JOIN teams th   ON th.team_id  = m.home_team_id
JOIN cities ch  ON ch.city_id  = th.city_id
JOIN teams ta   ON ta.team_id  = m.away_team_id
JOIN cities ca  ON ca.city_id  = ta.city_id
LEFT JOIN o     ON o.match_id  = m.match_id;

In [ ]:
%%writefile sql/06_analysis.sql
CREATE OR REPLACE VIEW v_distance_bands AS
WITH b AS (
    SELECT *,
           goal_diff - AVG(goal_diff) OVER (PARTITION BY league_id, season_label) AS gd_vs_league,
           CASE WHEN same_city        THEN '0 same city'
                WHEN travel_km < 100  THEN '1 <100 km'
                WHEN travel_km < 300  THEN '2 100-300'
                WHEN travel_km < 600  THEN '3 300-600'
                WHEN travel_km < 1200 THEN '4 600-1200'
                WHEN travel_km < 2500 THEN '5 1200-2500'
                ELSE                       '6 2500+' END AS band
    FROM match_facts
)
SELECT band, COUNT(*) AS n,
       AVG((result='H')::INT) AS home_win,
       AVG(gd_vs_league) AS gd_vs_league,
       STDDEV(gd_vs_league) / SQRT(COUNT(*)) AS se_gd,
       AVG(prob_home_fair) AS mkt_home_prob,
       AVG((result='H')::INT) FILTER (WHERE prob_home_fair IS NOT NULL) - AVG(prob_home_fair) AS home_win_minus_mkt
FROM b GROUP BY band;

CREATE OR REPLACE VIEW v_league_hfa AS
WITH t AS (
    SELECT *, match_date BETWEEN DATE '2020-03-15' AND DATE '2021-06-30' AS is_covid
    FROM match_facts WHERE start_year >= 2012
)
SELECT league_id, league_country, data_source,
       AVG(travel_km) FILTER (WHERE NOT is_covid)                   AS avg_km,
       COUNT(*) FILTER (WHERE NOT is_covid)                         AS n_normal,
       AVG(goal_diff) FILTER (WHERE NOT is_covid)                   AS hfa_normal,
       COUNT(*) FILTER (WHERE is_covid)                             AS n_covid,
       AVG(goal_diff) FILTER (WHERE is_covid)                       AS hfa_covid,
       AVG(goal_diff) FILTER (WHERE NOT is_covid)
         - AVG(goal_diff) FILTER (WHERE is_covid)                   AS covid_drop,
       STDDEV(goal_diff) FILTER (WHERE is_covid)
         / SQRT(COUNT(*) FILTER (WHERE is_covid))                   AS se_covid
FROM t GROUP BY ALL;

CREATE OR REPLACE VIEW v_market_by_period AS
SELECT CASE WHEN match_date <  DATE '2020-03-15' THEN '0 pre-COVID'
            WHEN match_date <  DATE '2020-09-01' THEN '1 Mar-Aug 2020'
            WHEN match_date <  DATE '2021-01-01' THEN '2 Sep-Dec 2020'
            WHEN match_date <= DATE '2021-06-30' THEN '3 Jan-Jun 2021'
            ELSE '4 post-COVID' END AS period,
       COUNT(*) AS n_odds,
       AVG((result='H')::INT) AS home_win,
       AVG(prob_home_fair) AS mkt_home_prob,
       AVG((result='H')::INT) - AVG(prob_home_fair) AS win_minus_mkt,
       1.96 * SQRT(AVG(prob_home_fair * (1 - prob_home_fair)) / COUNT(*)) AS ci95
FROM match_facts
WHERE start_year >= 2012 AND prob_home_fair IS NOT NULL
GROUP BY 1;

In [ ]:
for f in ['sql/05_views.sql', 'sql/06_analysis.sql']:
    con.execute(open(f).read())

display(con.sql("""
SELECT COUNT(*) n_rows, COUNT(travel_km) with_distance,
       SUM(same_city::INT) same_city, SUM(cross_border::INT) cross_border,
       COUNT(odds_source) with_odds,
       ROUND(MAX(ABS(prob_home_fair + prob_draw_fair + prob_away_fair - 1)), 6) max_prob_error,
       ROUND(MAX(travel_km)) max_km
FROM match_facts""").df())
con.execute("CHECKPOINT")

## 8. Findings

### 8.1 Travel distance
Goal difference is measured against each league-season's average, so league differences don't masquerade as distance effects.

**Result:** home teams do worse in same-city derbies and better against visitors from 2,500+ km, but the market's expected home-win rate rises with distance too, so actual results track the market in every band.

In [ ]:
display(con.sql("SELECT * FROM v_distance_bands ORDER BY band").df().round(4))

**Within-league check.** The pooled long-trip premium comes mostly from *which* leagues have long trips. Within the five longest-travel leagues, only Mexico shows a clear gradient, and that is concentrated in games against the two border clubs (Tijuana, Juárez). Altitude doesn't explain it: low-altitude hosts show the gap and high-altitude Guadalajara doesn't.

In [ ]:
display(con.sql("""
WITH b AS (
    SELECT league_id, result, prob_home_fair,
           goal_diff - AVG(goal_diff) OVER (PARTITION BY league_id, season_label) AS gd_vs_league,
           CASE WHEN same_city        THEN '0 same city'
                WHEN travel_km < 600  THEN '1 <600'
                WHEN travel_km < 1500 THEN '2 600-1500'
                ELSE                       '3 1500+' END AS band
    FROM match_facts WHERE league_id IN ('USA','RUS','BRA','CHN','MEX')
)
SELECT league_id, band, COUNT(*) n,
       ROUND(AVG(gd_vs_league), 3) gd_vs_league,
       ROUND(STDDEV(gd_vs_league) / SQRT(COUNT(*)), 3) se_gd,
       ROUND(AVG((result='H')::INT) - AVG(prob_home_fair), 4) win_minus_mkt,
       ROUND(SQRT(AVG(prob_home_fair * (1 - prob_home_fair)) / COUNT(prob_home_fair)), 4) se_mkt
FROM b GROUP BY 1, 2 ORDER BY 1, 2
""").df())

### 8.2 Between-league differences
On a common 2012+ window, European and global leagues have the same average home advantage. Log average travel explains about a fifth of the variation between leagues, driven largely by the USA.

In [ ]:
display(con.sql("""
SELECT ROUND(REGR_R2(hfa_normal, LN(avg_km)), 3)                      AS r2_log_distance,
       ROUND(REGR_SLOPE(hfa_normal, LN(avg_km)), 3)                   AS slope_per_log_km,
       ROUND(AVG(hfa_normal) FILTER (WHERE data_source = 'main'), 3)  AS main_hfa,
       ROUND(AVG(hfa_normal) FILTER (WHERE data_source = 'extra'), 3) AS extra_hfa
FROM v_league_hfa""").df())

### 8.3 Crowds (COVID natural experiment)
Matches from 15 Mar 2020 to 30 Jun 2021 are treated as empty-stadium games. The window is approximate (some leagues readmitted fans earlier), which biases the estimate toward zero, so the true effect is likely larger.

**Result:** within leagues, home advantage fell by 0.136 goals (95% CI ±0.036), about 40%. Per-league drops are noisy (one season each), so the pooled figure is the finding.

In [ ]:
display(con.sql("""
WITH t AS (SELECT league_id, goal_diff,
                  (match_date BETWEEN DATE '2020-03-15' AND DATE '2021-06-30')::INT AS covid
           FROM match_facts WHERE start_year >= 2012),
d AS (SELECT *, goal_diff - AVG(goal_diff) FILTER (WHERE covid = 0) OVER (PARTITION BY league_id) AS gd_dev
      FROM t)
SELECT covid, COUNT(*) n,
       ROUND(AVG(goal_diff), 3) raw_hfa,
       ROUND(AVG(gd_dev), 3) within_league_change,
       ROUND(1.96 * STDDEV(gd_dev) / SQRT(COUNT(*)), 3) ci95
FROM d GROUP BY 1 ORDER BY 1""").df())
display(con.sql("SELECT * FROM v_league_hfa ORDER BY covid_drop DESC").df().round(3))

### 8.4 Market pricing
Bookmakers cut expected home-win probability as soon as fans left, but not far enough: home teams underperformed the odds by 1.7 points in Sep–Dec 2020 (the only significant period). By 2021 the gap was gone. Home advantage has not fully returned since, and the market agrees.

In [ ]:
display(con.sql("SELECT * FROM v_market_by_period ORDER BY period").df().round(4))

## 9. Limitations
- Locations are city-level: same-city derbies have 0 km travel, and suburban grounds are approximated by their city.
- Odds-based analysis covers 2012+ only (closing odds); Pinnacle coverage ends partway through 2025.
- The COVID window is a single approximate date range across all leagues; summer-calendar leagues and early reopenings dilute it.
- Distance is straight-line, not actual travel time.
- Team quality is controlled only indirectly, through market odds.

## 10. Saving changes
Git is managed locally (PowerShell), not from Colab.
- **Notebook edits:** *File → Save a copy in GitHub* (repo `TylerWichman/global-home-advantage`, path `project_notebook.ipynb`).
- **SQL or Python edits:** the `%%writefile` cells above are the source; commit the matching files under `sql/` and `python/` locally.